In [12]:
import os
import csv
import time
import random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

print("CUDA available:", torch.cuda.is_available())

print(os.getcwd())


CUDA available: True
/home/users/ntu/lizh0106


In [49]:
# ====== Config (align with regular training pipeline paths) ======
# WORK_ROOT = Path.cwd().resolve().parent.parent  # notebook is under notebooks/legacy
#WORK_ROOT = Path("/scratch/users/ntu/lizh0106/nscc_work")
WORK_ROOT = Path("/home/users/ntu/lizh0106/scratch/nscc_work/")
FEATURES_ROOT = WORK_ROOT / "Processed_Features"

DATASET_NAME = "AGGC"
SCALE_DIR = "20x_512"
USE_L2 = False
COMBINE_2048 = True
SAVE = True
SEED = 42

# training debug knobs
N_SPLITS = 5
EPOCHS = 40
PATIENCE = 5
BATCH_SIZE = 8
NUM_WORKERS = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4

# if combine 2048, keep same model structure and only expand input dim
EXTRA_2048_REL = Path("AGGC_CENTER/AGGC_20x512/features.npy")

base = FEATURES_ROOT / DATASET_NAME
idx_path = base / SCALE_DIR / "index.csv"
feat_path = base / SCALE_DIR / "features.npy"

master_df = pd.read_csv(idx_path)
master_df["y"] = master_df["y"].astype(int)

run_tag = f"{'l2' if USE_L2 else 'nol2'}_{'with2048' if COMBINE_2048 else 'tileonly'}"
out_dir = WORK_ROOT / "Baseline_models/resultes" /"sampler_context_mha_aggc" / run_tag / "cv_results"
oof_dir = WORK_ROOT / "Baseline_models/resultes" /"sampler_context_mha_aggc" / run_tag / "oof_results"
out_dir.mkdir(parents=True, exist_ok=True)
oof_dir.mkdir(parents=True, exist_ok=True)

print("WORK_ROOT:", WORK_ROOT)
print("index:", idx_path)
print("features:", feat_path)
master_df.head(3)


WORK_ROOT: /home/users/ntu/lizh0106/scratch/nscc_work
index: /home/users/ntu/lizh0106/scratch/nscc_work/Processed_Features/AGGC/20x_512/index.csv
features: /home/users/ntu/lizh0106/scratch/nscc_work/Processed_Features/AGGC/20x_512/features.npy


,slide_id,y,start,length,n_tiles_read,h5_path
0,Subset1_Train_1,1,0,10339,10339,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...
1,Subset1_Train_10,3,10339,14579,14579,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...
2,Subset1_Train_100,3,24918,7562,7562,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...


In [65]:
# TCGA_BASE = Path("/scratch/users/ntu/lizh0106/nscc_work/Processed_Features/TGGA_PRAD_V2/TGGA_PRAD_V2_without_anno/TCGA_20x512/index.csv")

# TCGA_IDX_PATH = Path("/scratch/users/ntu/lizh0106/nscc_work/Processed_Features/TGGA_PRAD_V2/TGGA_PRAD_V2_without_anno/TCGA_20x512/index.csv")
# TCGA_FEAT_REL = FEATURES_ROOT / "TGGA_PRAD_V2/TGGA_PRAD_V2_without_anno/TCGA_20x512/features.npy"
# TCGA_EXTRA_2048_REL = FEATURES_ROOT / "TGGA_PRAD_V2/TGGA_PRAD_V2_without_anno_2048/TCGA_20x2048/features.npy"

TCGA_BASE = FEATURES_ROOT / "TCGA_PRAD_V2" 
TCGA_IDX_PATH = Path("/home/users/ntu/lizh0106/scratch/nscc_work/Processed_Features/TCGA_PRAD_V2/TCGA_PRAD_V2_without_anno/20x_512/index.csv")
TCGA_FEAT_REL = ("TCGA_PRAD_V2_without_anno", "20x_512", "features.npy")
TCGA_EXTRA_2048_REL = Path("Processed_Features/TCGA_PRAD_V2/TCGA_PRAD_V2_without_anno_2048/TCGA_20x2048/features.npy")

In [14]:
# out_dir =  "sampler_context_mha_aggc/l2_2048/cv_results"
# os.makedirs(out_dir, exist_ok=True) 

# oof_dir =  "sampler_context_mha_aggc/l2_2048/oof_results"
# os.makedirs(oof_dir, exist_ok=True)

idx_path = os.path.join(base, "20x_512", "index.csv")
feat_path = os.path.join(base, "20x_512", "features.npy")

master_df = pd.read_csv(idx_path)
master_df ["y"] = master_df ["y"].astype(int)
master_df.head(3)

,slide_id,y,start,length,n_tiles_read,h5_path
0,Subset1_Train_1,1,0,10339,10339,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...
1,Subset1_Train_10,3,10339,14579,14579,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...
2,Subset1_Train_100,3,24918,7562,7562,/home/users/ntu/lizh0106/scratch/nscc_work/AGG...


In [47]:
class PackedDataset(Dataset):
    """
    Slide-level bag dataset from memmap features.
    df columns needed: start, length, y, slide_id
    """
    def __init__(
        self,
        master_df,
        base_dir,
        feature_dim=1024,
        rel_path=("20x_512", "features.npy"),
        combine_2048=False,
        extra_2048_rel=None,
        use_l2=False,
        dtype=np.float32,
    ):
        self.df = master_df.reset_index(drop=True)
        self.D_main = int(feature_dim)
        self.combine_2048 = bool(combine_2048)
        self.extra_2048_rel = extra_2048_rel
        self.use_l2 = bool(use_l2)
        self.dtype = dtype

        self.feat_path = os.path.join(base_dir, *rel_path)
        if not os.path.exists(self.feat_path):
            raise FileNotFoundError(f"Main memmap not found: {self.feat_path}")

        self._mm_main = None
        self._mm_2048 = None
        self.D_2048 = 0

        if self.combine_2048:
            if self.extra_2048_rel is None:
                raise ValueError("combine_2048=True but extra_2048_rel is None")
            extra_path = os.path.join(base_dir, *tuple(extra_2048_rel))
            if not os.path.exists(extra_path):
                raise FileNotFoundError(f"2048 memmap not found: {extra_path}")
            self.extra_path = extra_path
            size = os.path.getsize(extra_path) // np.dtype(dtype).itemsize
            n_tiles = os.path.getsize(self.feat_path) // np.dtype(dtype).itemsize // self.D_main
            if n_tiles == 0 or size % n_tiles != 0:
                raise ValueError("Cannot infer 2048 feature dim from memmap file size.")
            self.D_2048 = int(size // n_tiles)

        self.D_total = self.D_main + self.D_2048

    def _l2_norm(self, x, eps=1e-8):
        return x / (np.linalg.norm(x, axis=1, keepdims=True) + eps)

    def _get_mm_main(self):
        if self._mm_main is None:
            self._mm_main = np.memmap(self.feat_path, dtype=self.dtype, mode="r").reshape(-1, self.D_main)
        return self._mm_main

    def _get_mm_2048(self):
        if not self.combine_2048:
            return None
        if self._mm_2048 is None:
            self._mm_2048 = np.memmap(self.extra_path, dtype=self.dtype, mode="r").reshape(-1, self.D_2048)
        return self._mm_2048

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        start = int(row["start"])
        length = int(row["length"])
        y = int(row["y"])
        slide_id = row["slide_id"]

        x_main = np.asarray(self._get_mm_main()[start:start + length], dtype=np.float32)

        if self.combine_2048:
            x_2048 = np.asarray(self._get_mm_2048()[start:start + length], dtype=np.float32)
            if self.use_l2:
                x_main = self._l2_norm(x_main)
                x_2048 = self._l2_norm(x_2048)
            x_np = np.concatenate([x_main, x_2048], axis=1)
        else:
            x_np = self._l2_norm(x_main) if self.use_l2 else x_main

        x_np = np.ascontiguousarray(x_np)
        return torch.from_numpy(x_np), torch.tensor(y, dtype=torch.long), slide_id, idx


def collate_varlen(batch):
    Xs, ys, ids, idx = zip(*batch)
    return list(Xs), torch.stack(ys, 0), list(ids), list(idx)


In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ContextMHA(nn.Module):
    def __init__(self, D=1024, H=4, num_classes=5, hidden_dim=512, p_drop=0.2):
        super().__init__()
        self.D = D
        self.H = H
        self.ln_x = nn.LayerNorm(D)

        # each head ONLY ONE w and b
        # We don't transpose here, as good practice is semantically have each row as attention head
        self.attn_w = nn.Parameter(torch.randn(H, D) * (1.0 / (D ** 0.5))) #normalize gradient, due to the attn mechanism
        self.attn_b = nn.Parameter(torch.zeros(H))

        self.classifier = nn.Sequential(
            nn.Linear(D * H, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, Xs, return_attn=False):
        """
        Xs: list of (N_i, D) OR a single tensor (N, D)
        """
        if isinstance(Xs, torch.Tensor):
            Xs = [Xs]

        device = self.attn_w.device
        slide_vecs = []
        attn_list = []  # store A for each slide

        for X in Xs:
            X = X.to(device).float()  # (N, D)
            X = self.ln_x(X)
            scores = torch.sigmoid(X @ self.attn_w.T + self.attn_b)  # (N, H)
            A = F.softmax(scores, dim=0)                             # (N, H)
            Z = A.T @ X                   # (H, D) # So this is a final score for a features that takes  all tiles into consider.

            slide_vecs.append(Z.reshape(-1))                         # (H*D,)
            if return_attn:
                attn_list.append(A.detach().cpu())                   # keep on CPU for saving/visual

        slide_mat = torch.stack(slide_vecs, dim=0)  # (B, H*D)
        logits = self.classifier(slide_mat)

        if return_attn:
            return logits, attn_list   # list length B, each (N_i, H)
        return logits



In [17]:
dataset = PackedDataset(
    master_df,
    base_dir=str(base),
    feature_dim=1024,
    rel_path=(SCALE_DIR, "features.npy"),
    combine_2048=COMBINE_2048,
    extra_2048_rel=tuple(EXTRA_2048_REL.parts),
    use_l2=USE_L2,
)
FEATURE_DIM = dataset.D_total

subset_idx = list(range(min(8, len(dataset))))
ds_small = Subset(dataset, subset_idx)
loader = DataLoader(
    ds_small,
    batch_size=4,
    shuffle=False,
    num_workers=min(2, NUM_WORKERS),
    pin_memory=True,
    collate_fn=collate_varlen,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, "| FEATURE_DIM:", FEATURE_DIM)

model = ContextMHA(D=FEATURE_DIM, H=4, num_classes=master_df['y'].nunique(), hidden_dim=512).to(device)
Xs, ys, ids, idx = next(iter(loader))
with torch.no_grad():
    logits = model(Xs)
print("Batch lens:", len(Xs), ys.shape, len(ids), "| logits:", logits.shape)


device: cuda | FEATURE_DIM: 2048


Batch lens: 4 torch.Size([4]) 4 | logits: torch.Size([4, 5])


In [18]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def make_class_weights(labels, num_classes):
    cnt = Counter(labels)
    total = sum(cnt.values())
    weights = [total / (num_classes * cnt.get(c, 1)) for c in range(num_classes)]
    return torch.tensor(weights, dtype=torch.float32)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for Xs, ys, _ids,idx in loader:
        ys = ys.to(device)
        optimizer.zero_grad()
        logits = model(Xs)
        loss = criterion(logits, ys)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        running_loss += loss.item() * ys.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device, num_classes=5, return_attn=False):
    model.eval()

    total_loss = 0.0
    all_y, all_pred = [], []

    
    dummy_weights = torch.ones(num_classes, device=device)
    criterion_eval = nn.CrossEntropyLoss(weight=dummy_weights)

    # 用于 OOF / 可视化的收集器
    all_ids = []
    all_preds = []
    all_probs = []
    attn_dict = {} if return_attn else None

    with torch.no_grad():
        for Xs, ys, _ids, idx in loader:
            ys = ys.to(device)

            out = model(Xs, return_attn=return_attn)
            if return_attn:
                logits, attn_list = out
            else:
                logits = out

            loss = criterion_eval(logits, ys)
            total_loss += loss.item() * ys.size(0)

            probs = F.softmax(logits, dim=1)          # (B, C)
            preds = probs.argmax(dim=1)               # (B,)

            # ===== metrics =====
            all_y.extend(ys.cpu().tolist())
            all_pred.extend(preds.cpu().tolist())

            # ===== OOF / per-sample outputs =====
            all_ids.extend(idx)
            all_preds.extend(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())

            if return_attn:
                for sid, A in zip(_ids, attn_list):
                    attn_dict[sid] = A  # A 已经是 cpu tensor（或你可以在这 detach）

    # ===== stack to numpy =====
    all_preds = np.asarray(all_preds)                 # (N,)
    all_probs = np.concatenate(all_probs, axis=0)     # (N, C)

    acc = accuracy_score(all_y, all_pred)
    macro_f1 = f1_score(all_y, all_pred, average="macro")
    report = classification_report(all_y, all_pred, digits=3)

    return (
        total_loss / len(loader.dataset),
        acc,
        macro_f1,
        report,
        all_ids,        # list[int] or list[str]
        all_preds,      # np.ndarray (N,)
        all_probs,      # np.ndarray (N, C)
        attn_dict,      # dict[id -> (N_i, H)] or None
    )




In [19]:
D = FEATURE_DIM
H = 4
labels = master_df["y"].tolist()
num_classes = len(set(labels))

from sklearn.model_selection import StratifiedKFold

set_seed(SEED)
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)


In [20]:
print("len labels: ",len(labels))

len labels:  187


In [21]:
model_tmp = ContextMHA(D=1024,H=H,num_classes=num_classes,hidden_dim=512,p_drop=0.2).to(device)

print(model_tmp)

# 参数统计
total_params = sum(p.numel() for p in model_tmp.parameters())
trainable_params = sum(p.numel() for p in model_tmp.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

del model_tmp

ContextMHA(
  (ln_x): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (classifier): Sequential(
    (0): Linear(in_features=4096, out_features=512, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=512, out_features=5, bias=True)
  )
)

Total parameters: 2,106,377
Trainable parameters: 2,106,377


In [22]:
fold_summaries = []
oof_pred = np.zeros(len(master_df), dtype=np.int64)
oof_prob = np.zeros((len(master_df), num_classes), dtype=np.float32)
all_attn = {}

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels), 1):
    print(f"\n===== Fold {fold} / {skf.n_splits} =====")
    ds_train = Subset(dataset, tr_idx)
    ds_valid = Subset(dataset, va_idx)

    train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                              collate_fn=collate_varlen, pin_memory=True)
    valid_loader = DataLoader(ds_valid, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                              collate_fn=collate_varlen, pin_memory=True)

    model = ContextMHA(D=D, H=H, num_classes=num_classes, hidden_dim=512, p_drop=0.2).to(device)

    y_train_fold = [labels[i] for i in tr_idx]
    class_weights = make_class_weights(y_train_fold, num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, verbose=True)

    best_f1, best_state, best_epoch = -1.0, None, 0
    bad = 0
    log_rows = []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        va_loss, va_acc, va_f1, va_rep, _, _, _, _ = evaluate(model, valid_loader, device, num_classes=num_classes)
        scheduler.step(va_f1)

        elapsed = time.time() - t0
        print(f"Epoch {epoch:02d} train_loss={tr_loss:.4f} val_loss={va_loss:.4f} val_acc={va_acc:.4f} val_macroF1={va_f1:.4f} ({elapsed:.1f}s)")

        log_rows.append({"fold": fold, "epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss,
                         "val_acc": va_acc, "val_f1": va_f1, "time_sec": elapsed})

        if va_f1 > best_f1 + 1e-4:
            best_f1 = va_f1
            best_epoch = epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"Early stopping at epoch {epoch}. Best epoch={best_epoch}")
                break

    log_path = out_dir / f"fold{fold}_log.csv"
    with open(log_path, "w", newline="") as fw:
        writer = csv.DictWriter(fw, fieldnames=log_rows[0].keys())
        writer.writeheader()
        writer.writerows(log_rows)

    model_path = out_dir / f"fold{fold}_best.pt"
    if best_state is not None:
        torch.save(best_state, model_path)
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    va_loss, va_acc, va_f1, va_rep, ids, preds, probs, attn_dict = evaluate(
        model, valid_loader, device, num_classes=num_classes, return_attn=True
    )

    oof_pred[ids] = preds
    oof_prob[ids] = probs
    all_attn.update(attn_dict)

    fold_summaries.append({
        "fold": fold,
        "best_epoch": best_epoch,
        "val_acc": va_acc,
        "val_f1": va_f1,
        "val_loss": va_loss,
        "best_model_path": str(model_path),
        "log_path": str(log_path),
    })

df_summary = pd.DataFrame(fold_summaries)
df_summary.to_csv(out_dir / "cv_summary.csv", index=False)

print("\n==== 5-Fold Summary ====")
print(df_summary[["fold", "val_acc", "val_f1", "best_epoch"]])
print(f"\nMean F1={df_summary['val_f1'].mean():.4f}, Std={df_summary['val_f1'].std():.4f}")



===== Fold 1 / 5 =====
Epoch 01 train_loss=5.5306 val_loss=3.0039 val_acc=0.4474 val_macroF1=0.2291 (23.9s)
Epoch 02 train_loss=5.0447 val_loss=2.3611 val_acc=0.3684 val_macroF1=0.2339 (6.7s)
Epoch 03 train_loss=1.4081 val_loss=1.4416 val_acc=0.5000 val_macroF1=0.4328 (6.8s)
Epoch 04 train_loss=1.4755 val_loss=1.5299 val_acc=0.3684 val_macroF1=0.1678 (6.5s)
Epoch 05 train_loss=1.5219 val_loss=1.6043 val_acc=0.4474 val_macroF1=0.2702 (6.5s)
Epoch 06 train_loss=1.2457 val_loss=1.7815 val_acc=0.2895 val_macroF1=0.2445 (6.5s)
Epoch 07 train_loss=1.3590 val_loss=1.6742 val_acc=0.2895 val_macroF1=0.2368 (6.5s)
Epoch 08 train_loss=1.0574 val_loss=1.3031 val_acc=0.4474 val_macroF1=0.3544 (6.4s)
Early stopping at epoch 8. Best epoch=3

===== Fold 2 / 5 =====
Epoch 01 train_loss=6.6709 val_loss=3.4528 val_acc=0.2632 val_macroF1=0.1352 (6.4s)
Epoch 02 train_loss=3.1236 val_loss=2.3184 val_acc=0.4211 val_macroF1=0.1638 (6.4s)
Epoch 03 train_loss=2.3103 val_loss=1.6395 val_acc=0.3158 val_macroF1=0

In [23]:
attn_np_dict = {
    k: v.detach().cpu().numpy()
    for k, v in all_attn.items()
}

In [24]:
if SAVE:
    attn_np_dict = {k: v.detach().cpu().numpy() for k, v in all_attn.items()}
    np.save(oof_dir / "aggc_oof_slide_pred.npy", oof_pred)
    np.save(oof_dir / "aggc_oof_slide_prob.npy", oof_prob)
    np.savez(oof_dir / "attn_map_aggc.npz", **attn_np_dict)


In [25]:
oof_pred = np.load(oof_dir / "aggc_oof_slide_pred.npy")
oof_prob = np.load(oof_dir / "aggc_oof_slide_prob.npy")


In [26]:
print("Slide-level OOF classification report")
print(classification_report(master_df["y"].values, oof_pred, digits=3))
print("Macro F1:", f1_score(master_df["y"].values, oof_pred, average="macro"))
print("Accuracy:", accuracy_score(master_df["y"].values, oof_pred))
print("Confusion matrix:", confusion_matrix(master_df["y"].values, oof_pred))


Slide-level OOF classification report
              precision    recall  f1-score   support

           0      0.429     0.545     0.480        11
           1      0.656     0.782     0.713        78
           2      0.641     0.379     0.476        66
           3      0.154     0.200     0.174        10
           4      0.500     0.636     0.560        22

    accuracy                          0.578       187
   macro avg      0.476     0.509     0.481       187
weighted avg      0.592     0.578     0.569       187

Macro F1: 0.48071076241327954
Accuracy: 0.5775401069518716
Confusion matrix: [[ 6  4  1  0  0]
 [ 6 61  6  2  3]
 [ 0 26 25  6  9]
 [ 2  2  2  2  2]
 [ 0  0  5  3 14]]


In [64]:
# TCGA inference: load 5 CV models -> predict slide-level probabilities -> mean ensemble


def predict_only(model, loader, device):
    model.eval()
    all_ids, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for Xs, _ys, _ids, idx in loader:
            logits = model(Xs)
            probs = F.softmax(logits, dim=1)
            preds = probs.argmax(dim=1)
            all_ids.extend(idx)
            all_preds.extend(preds.cpu().numpy())
            all_probs.append(probs.cpu().numpy())
    return np.asarray(all_ids), np.asarray(all_preds), np.concatenate(all_probs, axis=0)


tcga_df = pd.read_csv(TCGA_IDX_PATH)
if "y" not in tcga_df.columns:
    tcga_df["y"] = 0
else:
    tcga_df["y"] = tcga_df["y"].fillna(0).astype(int)

if "slide_id" not in tcga_df.columns:
    raise KeyError("TCGA index.csv must contain 'slide_id' column for slide-level saving.")

tcga_dataset = PackedDataset(
    tcga_df,
    base_dir=str(TCGA_BASE),
    feature_dim=1024,
    rel_path=TCGA_FEAT_REL,
    combine_2048=COMBINE_2048,
    extra_2048_rel=tuple(TCGA_EXTRA_2048_REL.parts),
    use_l2=USE_L2,
)

if tcga_dataset.D_total != D:
    raise ValueError(f"TCGA feature dim {tcga_dataset.D_total} != training dim {D}")

tcga_loader = DataLoader(
    tcga_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_varlen,
    pin_memory=True,
)

fold_model_paths = [out_dir / f"fold{i}_best.pt" for i in range(1, N_SPLITS + 1)]
for mp in fold_model_paths:
    if not mp.exists():
        raise FileNotFoundError(f"Missing fold checkpoint: {mp}")

tcga_fold_probs = []
for i, model_path in enumerate(fold_model_paths, 1):
    print(f"[TCGA] Running model {i}/{len(fold_model_paths)}: {model_path}")
    model = ContextMHA(D=D, H=H, num_classes=num_classes, hidden_dim=512, p_drop=0.2).to(device)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)

    ids, preds, probs = predict_only(model, tcga_loader, device)
    tcga_fold_probs.append(probs)

tcga_prob_stack = np.stack(tcga_fold_probs, axis=0)  # (5, n_slide, n_classes)
tcga_prob_mean = tcga_prob_stack.mean(axis=0)
tcga_pred_mean = tcga_prob_mean.argmax(axis=1)

if SAVE:
    np.save(oof_dir / "tcga_slide_prob_per_fold.npy", tcga_prob_stack)
    np.save(oof_dir / "tcga_slide_prob_mean.npy", tcga_prob_mean)
    np.save(oof_dir / "tcga_slide_pred_mean.npy", tcga_pred_mean)

    tcga_out_df = pd.DataFrame({
        "slide_id": tcga_df["slide_id"].values,
        "pred_label": tcga_pred_mean,
    })
    for c in range(num_classes):
        tcga_out_df[f"proba_class_{c}"] = tcga_prob_mean[:, c]
    tcga_out_df.to_csv(oof_dir / "tcga_slide_predictions.csv", index=False)

print("TCGA prob stack:", tcga_prob_stack.shape)
print("TCGA prob mean:", tcga_prob_mean.shape)
print("TCGA pred labels:", tcga_pred_mean.shape)



FileNotFoundError: Main memmap not found: /home/users/ntu/lizh0106/scratch/nscc_work/Processed_Features/TCGA_PRAD_V2/TGGA_PRAD_V2_without_anno/20x_512/features.npy

- Find a corrected classified sample
- Define a method to extract thumbnail
- Define 2 methods to visualiza map on image  

In [ ]:
"/home/users/ntu/lizh0106/scratch/nscc_work/Processed_Features/TCGA_PRAD_V2/TGGA_PRAD_V2_without_anno/20x_512/features.npy"
"/home/users/ntu/lizh0106/scratch/nscc_work/Processed_Features/TGGA_PRAD_V2/TGGA_PRAD_V2_without_anno/20x_512/features.npy"